# GRAPE Optimal Control Workflow

This notebook shows the full direct-control workflow now supported in `cqed_sim`:

- build a `ControlProblem` from an existing model
- solve it with the closed-system `GrapeSolver`
- export the optimized schedule into standard runtime pulses
- replay the result through the simulator
- evaluate the same optimized schedule under Lindblad noise with `evaluate_with_simulator(...)`
- run the benchmark harness for larger validation cases

The important convention split is:

- optimization is currently done under closed-system assumptions
- noisy replay is a simulator-backed evaluation path, not a noisy-gradient optimizer

In [1]:
from pathlib import Path
import sys

import numpy as np
import qutip as qt

for candidate in (Path.cwd(), *Path.cwd().parents):
    if (candidate / "cqed_sim").exists():
        repo_root = candidate
        break
else:
    raise RuntimeError("Could not locate the repository root containing cqed_sim.")

if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

from cqed_sim import (
    ControlEvaluationCase,
    DispersiveTransmonCavityModel,
    FrameSpec,
    GrapeConfig,
    GrapeSolver,
    ModelControlChannelSpec,
    NoiseSpec,
    PiecewiseConstantTimeGrid,
    SequenceCompiler,
    SimulationConfig,
    UnitaryObjective,
    build_control_problem_from_model,
    simulate_sequence,
)
from cqed_sim.unitary_synthesis import Subspace

_ = (
    np,
    qt,
    ControlEvaluationCase,
    DispersiveTransmonCavityModel,
    FrameSpec,
    GrapeConfig,
    GrapeSolver,
    ModelControlChannelSpec,
    NoiseSpec,
    PiecewiseConstantTimeGrid,
    SequenceCompiler,
    SimulationConfig,
    UnitaryObjective,
    build_control_problem_from_model,
    simulate_sequence,
    Subspace,
)

In [2]:
def rotation_y(theta: float) -> np.ndarray:
    return np.array(
        [
            [np.cos(theta / 2.0), -np.sin(theta / 2.0)],
            [np.sin(theta / 2.0), np.cos(theta / 2.0)],
        ],
        dtype=np.complex128,
    )


model = DispersiveTransmonCavityModel(
    omega_c=2.0 * np.pi * 5.0e9,
    omega_q=2.0 * np.pi * 6.0e9,
    alpha=0.0,
    chi=0.0,
    kerr=0.0,
    n_cav=3,
    n_tr=2,
)
frame = FrameSpec(omega_c_frame=model.omega_c, omega_q_frame=model.omega_q)
storage_logical = Subspace.custom(
    full_dim=6,
    indices=(0, 1),
    labels=("|g,0>", "|g,1>"),
)

time_grid = PiecewiseConstantTimeGrid.uniform(steps=8, dt_s=10.0e-9)
problem = build_control_problem_from_model(
    model,
    frame=frame,
    time_grid=time_grid,
    channel_specs=(
        ModelControlChannelSpec(
            name="storage",
            target="storage",
            quadratures=("Q",),
            amplitude_bounds=(-1.0e8, 1.0e8),
            export_channel="storage",
        ),
    ),
    objectives=(
        UnitaryObjective(
            target_operator=rotation_y(np.pi / 2.0),
            subspace=storage_logical,
            ignore_global_phase=True,
            name="storage_y90",
        ),
    ),
    metadata={"tutorial": "grape_optimal_control_workflow"},
)

initial_schedule = np.full((problem.n_controls, problem.n_slices), 8.0e6, dtype=float)
problem

ControlProblem(parameterization=PiecewiseConstantParameterization(time_grid=PiecewiseConstantTimeGrid(step_durations_s=(1e-08, 1e-08, 1e-08, 1e-08, 1e-08, 1e-08, 1e-08, 1e-08), t0_s=0.0), control_terms=(ControlTerm(name='storage_Q', operator=array([[0.+0.j        , 0.+1.j        , 0.+0.j        , 0.+0.j        ,
        0.+0.j        , 0.+0.j        ],
       [0.-1.j        , 0.+0.j        , 0.+1.41421356j, 0.+0.j        ,
        0.+0.j        , 0.+0.j        ],
       [0.+0.j        , 0.-1.41421356j, 0.+0.j        , 0.-0.j        ,
        0.+0.j        , 0.+0.j        ],
       [0.+0.j        , 0.+0.j        , 0.-0.j        , 0.+0.j        ,
        0.+1.j        , 0.+0.j        ],
       [0.+0.j        , 0.+0.j        , 0.+0.j        , 0.-1.j        ,
        0.+0.j        , 0.+1.41421356j],
       [0.+0.j        , 0.+0.j        , 0.+0.j        , 0.+0.j        ,
        0.-1.41421356j, 0.+0.j        ]]), amplitude_bounds=(-100000000.0, 100000000.0), export_channel='storage', drive_

In [3]:
solver = GrapeSolver(GrapeConfig(maxiter=80, seed=7, random_scale=0.2))
result = solver.solve(problem, initial_schedule=initial_schedule)

{
    "success": result.success,
    "message": result.message,
    "objective_value": result.objective_value,
    "nominal_fidelity": result.metrics["nominal_fidelity"],
    "max_abs_amplitude": result.schedule.max_abs_amplitude(),
    "iterations": result.optimizer_summary["nit"],
}

{'success': True,
 'message': 'CONVERGENCE: NORM_OF_PROJECTED_GRADIENT_<=_PGTOL',
 'objective_value': 0.28894017107948067,
 'nominal_fidelity': 0.8580094884711515,
 'max_abs_amplitude': 4684368.958076183,
 'iterations': 5}

In [4]:
pulses, drive_ops, pulse_meta = result.to_pulses()
compiler = SequenceCompiler(dt=1.0e-9)
compiled = compiler.compile(pulses, t_end=problem.time_grid.duration_s)

initial_state = model.basis_state(0, 0)
runtime = simulate_sequence(
    model,
    compiled,
    initial_state,
    drive_ops,
    config=SimulationConfig(frame=frame),
)

target_state = qt.Qobj(
    np.array([1.0 / np.sqrt(2.0), 1.0 / np.sqrt(2.0), 0.0, 0.0, 0.0, 0.0], dtype=np.complex128),
    dims=initial_state.dims,
)

{
    "pulse_count": len(pulses),
    "pulse_export_summary": pulse_meta["channels"],
    "runtime_replay_fidelity": float(qt.metrics.fidelity(runtime.final_state, target_state)),
}

{'pulse_count': 8,
 'pulse_export_summary': {'storage': {'max_abs_amp': 4684368.958076183,
   'nonzero_slices': 8}},
 'runtime_replay_fidelity': 0.9054363186031267}

In [5]:
nominal_replay = result.evaluate_with_simulator(
    problem,
    model=model,
    frame=frame,
    compiler_dt_s=1.0e-9,
)

noisy_replay = result.evaluate_with_simulator(
    problem,
    cases=(
        ControlEvaluationCase(
            model=model,
            label="lossy_storage",
            frame=frame,
            noise=NoiseSpec(kappa=2.0e5),
            compiler_dt_s=1.0e-9,
        ),
    ),
)

{
    "closed_system_replay_fidelity": nominal_replay.metrics["aggregate_fidelity"],
    "noisy_replay_fidelity": noisy_replay.metrics["aggregate_fidelity"],
    "noisy_replay_case": noisy_replay.member_reports[0].label,
    "noisy_replay_objective": noisy_replay.member_reports[0].objective_reports[0].name,
}

{'closed_system_replay_fidelity': 0.842048808510462,
 'noisy_replay_fidelity': 0.8391811145678141,
 'noisy_replay_case': 'lossy_storage',
 'noisy_replay_objective': 'storage_y90'}

## Benchmark Harness

For larger validation sweeps, use the repository benchmark script:

- `benchmarks/run_optimal_control_benchmarks.py`

It can run small and larger state-transfer cases, a leakage-aware subspace case, and a reduced-versus-three-mode comparison. The benchmark output includes optimizer runtime, final fidelity, replay fidelity, penalty totals, and model-size metadata.

In [6]:
from subprocess import run

benchmark_output = repo_root / "outputs" / "tutorial_grape_benchmark.json"
command = [
    sys.executable,
    str(repo_root / "benchmarks" / "run_optimal_control_benchmarks.py"),
    "--suite",
    "smoke",
    "--maxiter",
    "20",
    "--output",
    str(benchmark_output),
]

print("Command:")
print(" ".join(command))

completed = run(command, capture_output=True, text=True, check=True)
print(completed.stdout)

benchmark_output

Command:
C:\Users\jl82323\AppData\Local\Programs\Python\Python312\python.exe C:\Users\jl82323\Box\Shyam Shankar Quantum Circuits Group\Users\Users_JianJun\cQED_simulation\benchmarks\run_optimal_control_benchmarks.py --suite smoke --maxiter 20 --output C:\Users\jl82323\Box\Shyam Shankar Quantum Circuits Group\Users\Users_JianJun\cQED_simulation\outputs\tutorial_grape_benchmark.json


{
  "suite": "smoke",
  "results": [
    {
      "case": "small_state",
      "backend": "grape",
      "seed": 7,
      "configuration": {
        "slices": 2,
        "duration_s": 4e-08,
        "target_type": "state_transfer",
        "model_regime": "two_mode",
        "aggregate_mode": "mean",
        "amplitude_penalty": 0.0,
        "slew_penalty": 0.0,
        "leakage_weight": 0.0,
        "robust_frequency_shift_rad_s": null,
        "compiler_dt_s": null,
        "max_step_s": null
      },
      "model": {
        "type": "DispersiveTransmonCavityModel",
        "subsystem_dims": [
          2,
          1
        ],
        "full_dim": 2
      },
      "solve": {
        "success": true,
        "message": "CONVERGENCE: NORM_OF_PROJECTED_GRADIENT_<=_PGTOL",
        "runtime_s": 0.0034382000158075243,
        "iteration_count": 0,
        "function_evaluations": 1,
        "objective_value": 0.0,
        "nominal_fidelity": 1.0,
        "control_penalty_total": 0.0,
      

WindowsPath('C:/Users/jl82323/Box/Shyam Shankar Quantum Circuits Group/Users/Users_JianJun/cQED_simulation/outputs/tutorial_grape_benchmark.json')

## Interpretation

Use the three views together:

- `result.metrics["nominal_fidelity"]` for the optimizer's closed-system objective summary
- `evaluate_with_simulator(...)` for simulator-backed replay under nominal or noisy conditions
- `benchmarks/run_optimal_control_benchmarks.py` for larger regression and scaling checks

That separation keeps the current GRAPE implementation honest about what is optimized versus what is only evaluated.